In [1]:
from dataclasses import dataclass
@dataclass
class WikipediaPage:
  title: str
  url: str
  relevant_text: str = None

In [2]:
# prompt: write code to search wikipedia for a topic and return titles and urls of the pages found
def search_wikipedia(query: str):
  import wikipedia
  wikipedia.set_lang("en")
  results = wikipedia.search(query)
  pages = []
  for title in results:
    try:
      page = wikipedia.page(title)
      pages.append(WikipediaPage(title=page.title,
                                 url=page.url))
    except wikipedia.exceptions.DisambiguationError as e:
      print(f"Disambiguation error for '{title}': {e}")
      # Handle disambiguation, e.g., choose the first option or skip
      # Here, we're skipping the disambiguation pages
      continue
    except wikipedia.exceptions.PageError:
      print(f"Page not found for '{title}'")
      continue
  return pages




In [3]:
# Example usage
query = "What were the causes of the Liberian civil war?"
pages = search_wikipedia(query)
pages

Page not found for 'Burundian Civil War'


[WikipediaPage(title='First Liberian Civil War', url='https://en.wikipedia.org/wiki/First_Liberian_Civil_War', relevant_text=None),
 WikipediaPage(title='Nigeria', url='https://en.wikipedia.org/wiki/Nigeria', relevant_text=None),
 WikipediaPage(title='Sudanese civil war (2023–present)', url='https://en.wikipedia.org/wiki/Sudanese_civil_war_(2023%E2%80%93present)', relevant_text=None),
 WikipediaPage(title='Causes of World War I', url='https://en.wikipedia.org/wiki/Causes_of_World_War_I', relevant_text=None),
 WikipediaPage(title='Americo-Liberian people', url='https://en.wikipedia.org/wiki/Americo-Liberian_people', relevant_text=None),
 WikipediaPage(title='Algerian Civil War', url='https://en.wikipedia.org/wiki/Algerian_Civil_War', relevant_text=None),
 WikipediaPage(title='History of Liberia', url='https://en.wikipedia.org/wiki/History_of_Liberia', relevant_text=None),
 WikipediaPage(title='List of war crimes', url='https://en.wikipedia.org/wiki/List_of_war_crimes', relevant_text=Non

In [10]:
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider


ollama_provider = OpenAIProvider(
    base_url="http://localhost:11434/v1",
    api_key="ollama"  # A placeholder key is required by the client
)

# 2. Initialize the model with your local pulled model name
ollama_model = OpenAIChatModel(
    model_name="gemma4",
    provider=ollama_provider
)


In [11]:
async def rank_pages(query: str, pages: list[WikipediaPage]) -> list[WikipediaPage]:
  agent = Agent(model=ollama_model, output_type=list[WikipediaPage])
  prompt = f"""Rank these Wikipedia pages by relevance to the query: "{query}".
  Pages: {pages}"""
  response = await agent.run(prompt)
  return response.output


In [12]:
# Example usage
ranked_pages = (await rank_pages(query, pages))[:2] # top 2
ranked_pages


[WikipediaPage(title='First Liberian Civil War', url='https://en.wikipedia.org/wiki/First_Liberian_Civil_War', relevant_text=None),
 WikipediaPage(title='History of Liberia', url='https://en.wikipedia.org/wiki/History_of_Liberia', relevant_text=None)]

In [15]:
async def add_relevant_text(query: str, page: WikipediaPage):
  agent = Agent(model=ollama_model, output_type=str)
  prompt = f"""
  Read {page.url} and extract the text relevant to the following query.
  Return only the relevant text without any preamble.
  {query}
  """
  response = await agent.run(prompt)
  page.relevant_text = response.output

In [16]:
await add_relevant_text(query, ranked_pages[0])

In [18]:
async def synthesize_answer(query: str, page: WikipediaPage) -> str:
  agent = Agent(model=ollama_model, output_type=str)
  prompt = f"""
  Answer the following query based on the given information.
  Query:
  {query}

  Relevant information:
  {page.relevant_text }
  """
  response = await agent.run(prompt)
  return response.output

In [19]:
answer = await synthesize_answer(query, ranked_pages[0])
print(answer)

The Liberian civil war was caused by a confluence of deep-rooted internal struggles and external influences. According to the provided information, the key contributing factors include:

**1. Political Instability and Corruption:**
*   The country had long suffered cycles of political upheaval and corruption within its ruling elite.
*   This instability eroded public trust and weakened institutional strength, often leading regime changes or leadership conflicts to be settled violently.

**2. Social Divisions (Ethnic/Regional Tensions):**
*   Deep-seated tensions existed between different ethnic groups and geographical regions. These divisions were frequently exploited by political factions for personal advantage.

**3. Economic Decline:**
*   Economic mismanagement, combined with the decline of Liberia's major commodity export markets, led to widespread poverty and resource scarcity. This hardship fueled desperation among various armed groups.

**4. External Interference:**
*   Neighbo